# Project Lazarus — Whisper Large-V3 Translation Server

Run this notebook to expose a public ngrok tunnel that Project Lazarus uses for the highest accuracy multilingual audio transcription and translation.
It uses **Whisper Large-V3** (the most accurate Whisper model) to transcribe and automatically **translate** English, Malayalam, Hindi, etc., into English text.

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells in order
3. Copy the `ngrok tunnel` URL printed at the end
4. Paste it into Project Lazarus: **Settings → Engine → Google Colab tab → Colab URL**

In [ ]:
# Cell 1: Install dependencies
!pip install faster-whisper flask flask-cors pyngrok nest-asyncio -q
print("Dependencies installed successfully!")

In [ ]:
# Cell 2: Load Whisper Large-V3 (highest accuracy) on GPU
from faster_whisper import WhisperModel

print("Loading Whisper Large-V3 (float16) on GPU...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Whisper Large-V3 loaded successfully!")

In [ ]:
# Cell 3: Define HTTP server with Whisper Large-V3 translation to English
import io
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/ping", methods=["GET"])
def ping():
    return jsonify({"status": "ok"}), 200

@app.route("/transcribe", methods=["POST"])
def transcribe():
    if "file" not in request.files:
        return jsonify({"error": "No file provided"}), 400

    file = request.files["file"]
    audio_bytes = file.read()

    try:
        # Transcribe and translate audio directly to English using Whisper's built-in translation task
        segments, info = model.transcribe(
            io.BytesIO(audio_bytes),
            task="translate",
            beam_size=5,
            word_timestamps=True
        )
        translated_text = " ".join(seg.text for seg in segments).strip()
    except Exception as e:
        return jsonify({"error": f"Whisper translation failed: {str(e)}"}), 500

    return jsonify({"text": translated_text}), 200

print("Server defined")

In [ ]:
# Cell 4: Authenticate and start ngrok tunnel
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3Et6J2xAG2JUpx2AXXnbc90PJAj_6tTtMY4gByuuUYG2pJibf"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000, bind_tls=True).public_url
print(f"ngrok tunnel: {public_url}")
print(f"Ping test:    {public_url}/ping")

In [ ]:
# Cell 5: Start Flask server
import nest_asyncio

nest_asyncio.apply()
app.run(port=5000, host="0.0.0.0")